# SKU Mapping: Maxab vs Competitor Scraped Data
**Approach**: Arabic text normalization → Brand filtering → Token-based fuzzy matching → Price validation

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from rapidfuzz import fuzz, process
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

In [ ]:
maxab = pd.read_excel(r'd:\scripts\Mustafa\Mapping\maxab_raw_data.xlsx')
scrapped = pd.read_excel(r'd:\scripts\Mustafa\Mapping\scrapped_raw_data.xlsx')
print(f"Maxab: {maxab.shape[0]} rows, {maxab['PRODUCT_ID'].nunique()} unique products")
print(f"Scrapped: {scrapped.shape[0]} rows from {scrapped['SOURCE_APP'].nunique()} apps")
print(f"\nScrapped per app:")
print(scrapped['SOURCE_APP'].value_counts().to_string())

## Arabic Text Normalization (State-of-the-Art)

In [ ]:
ARABIC_DIACRITICS = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]')
TATWEEL = '\u0640'

ALEF_VARIANTS = {
    '\u0622': '\u0627',  # آ → ا
    '\u0623': '\u0627',  # أ → ا
    '\u0625': '\u0627',  # إ → ا
    '\u0671': '\u0627',  # ٱ → ا
}

MEASUREMENT_NORMALIZATION = {
    'جرام': 'جم', 'غرام': 'جم', 'غم': 'جم', 'جرم': 'جم', 'g': 'جم', 'gm': 'جم', 'gr': 'جم',
    'كيلو': 'كجم', 'كيلوجرام': 'كجم', 'كيلوغرام': 'كجم', 'kg': 'كجم',
    'ملل': 'مل', 'ملي': 'مل', 'ميلي': 'مل', 'ml': 'مل',
    'لتر': 'لتر', 'ليتر': 'لتر', 'l': 'لتر',
    'سنتي': 'سم', 'سنتيمتر': 'سم', 'cm': 'سم',
    'قطعه': 'قطعة', 'حبه': 'حبة', 'عبوه': 'عبوة',
    'فتله': 'فتلة', 'بكره': 'بكرة', 'حفاضه': 'حفاضة',
}

# Convert all size units to base: grams (weight), milliliters (volume), millimeters (length)
SIZE_TO_BASE = {
    'جم': ('weight', 1.0),
    'كجم': ('weight', 1000.0),
    'مل': ('volume', 1.0),
    'لتر': ('volume', 1000.0),
    'سم': ('length', 10.0),
}

EASTERN_TO_WESTERN = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')


def normalize_arabic(text):
    """State-of-the-art Arabic text normalization for product matching."""
    if not isinstance(text, str) or not text.strip():
        return ''
    
    text = text.strip()
    text = ARABIC_DIACRITICS.sub('', text)
    text = text.replace(TATWEEL, '')
    
    for variant, canonical in ALEF_VARIANTS.items():
        text = text.replace(variant, canonical)
    
    text = text.replace('\u0629', '\u0647')  # ة → ه
    text = text.replace('\u0649', '\u064A')  # ى → ي
    
    text = text.translate(EASTERN_TO_WESTERN)
    
    text = text.lower()
    
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    
    tokens = text.split()
    tokens = [MEASUREMENT_NORMALIZATION.get(t, t) for t in tokens]
    
    text = ' '.join(tokens)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


def extract_size_info(text):
    """Extract numeric size/weight/volume from product name."""
    normalized = normalize_arabic(text)
    patterns = [
        r'(\d+(?:\.\d+)?)\s*(كجم|جم|مل|لتر|سم)',
    ]
    matches = []
    for pat in patterns:
        matches.extend(re.findall(pat, normalized))
    return matches


def normalize_size_to_base(value, unit):
    """Convert size to base unit (grams/ml/mm) for comparison."""
    val = float(value)
    if unit in SIZE_TO_BASE:
        dimension, multiplier = SIZE_TO_BASE[unit]
        return dimension, val * multiplier
    return None, None


def sizes_compatible(text_a, text_b, tolerance=0.15):
    """Hard filter: if both products have extractable sizes, they MUST be within tolerance.
    Returns: True (sizes match), False (sizes conflict), None (can't determine)."""
    sizes_a = extract_size_info(text_a)
    sizes_b = extract_size_info(text_b)
    
    if not sizes_a or not sizes_b:
        return None
    
    for (val_a, unit_a) in sizes_a:
        dim_a, base_a = normalize_size_to_base(val_a, unit_a)
        if dim_a is None:
            continue
        for (val_b, unit_b) in sizes_b:
            dim_b, base_b = normalize_size_to_base(val_b, unit_b)
            if dim_b is None:
                continue
            if dim_a == dim_b:
                if base_a == 0 and base_b == 0:
                    return True
                max_val = max(base_a, base_b)
                if max_val > 0:
                    diff = abs(base_a - base_b) / max_val
                    return diff <= tolerance
    return None


print("Test normalization:")
tests = [
    "إندومى - نودلز - لحمه صغير - 70 جرام",
    "اندومي بطعم الفراخ + 4 اكياس مجانا - 70 جم",
    "بيبسي مشروب غازي - 2.5 لتر",
    "بيبسي هضبه - 250 مل",
    "تايد اوتوماتيك برائحة داوني - 900 جم",
]
for t in tests:
    print(f"  {t}")
    print(f"  → {normalize_arabic(t)}")
    sizes = extract_size_info(t)
    for v, u in sizes:
        dim, base = normalize_size_to_base(v, u)
        print(f"  → size: {v} {u} = {base} base ({dim})")
    print()

print("Size compatibility tests:")
print(f"  2.5 لتر vs 250 مل: {sizes_compatible('بيبسي - 2.5 لتر', 'بيبسي - 250 مل')}")
print(f"  70 جم vs 70 جم: {sizes_compatible('اندومي - 70 جم', 'اندومي - 70 جرام')}")
print(f"  900 جم vs 1 كجم: {sizes_compatible('تايد - 900 جم', 'تايد - 1 كجم')}")
print(f"  400 جم vs 400 جم: {sizes_compatible('مكرونة - 400 جم', 'مكرونة - 400 جم')}")
print(f"  1 لتر vs 1 لتر: {sizes_compatible('زيت - 1 لتر', 'زيت - 1 لتر')}")
print(f"  500 مل vs 1 لتر: {sizes_compatible('ماء - 500 مل', 'ماء - 1 لتر')}")

## Unit Type Normalization

In [ ]:
UNIT_TYPE_CANONICAL = {
    # Carton family
    'carton': 'carton', 'كرتونه': 'carton', 'كرتون': 'carton', 'box': 'carton',
    # Shrink family
    'shrink': 'shrink', 'شرينك': 'shrink', 'شرنك': 'shrink', 'sherink': 'shrink',
    # Box/علبه
    'علبه': 'box',
    # Item/عبوه
    'item': 'item', 'عبوه': 'item',
    # Piece
    'piece': 'piece', 'قطعه': 'piece',
    # Bala
    'pala': 'bala', 'باله': 'bala',
    # Packet
    'packet': 'packet', 'باكت': 'packet',
    # Jar
    'jar': 'jar', 'برطمان': 'jar',
    # Kees/bag
    'kees': 'kees', 'كيس': 'kees', 'bag': 'kees',
    # Jerry can
    'jerry can': 'jerrycan', 'جركن': 'jerrycan',
    # Roll
    'roll': 'roll', 'رول': 'roll',
    # Shikara
    'shikara': 'shikara', 'shekara': 'shikara', 'شيكاره': 'shikara', 'شكاره': 'shikara',
    # Strip
    'strip': 'strip', 'شريط': 'strip',
    # Mat
    'mat': 'mat', 'حصيره': 'mat',
    # Cans
    'cans': 'cans', 'can': 'cans', 'كانز': 'cans',
    # Offer/bundle
    'offer': 'offer', 'عرض': 'offer', 'تجميعه': 'offer',
    # Half carton
    '1/2 carton': 'half_carton', 'نص كرتونه': 'half_carton',
    # Quarter carton
    '1/4 carton': 'quarter_carton', 'ربع كرتونه': 'quarter_carton',
    # 3/4 carton
    '4/3 carton': '3q_carton', 'تلت ارباع كرتونه': '3q_carton',
    # Others
    'pallet': 'pallet', 'بالته': 'pallet',
    'rabta': 'rabta', 'ربطه': 'rabta',
    'lafa': 'lafa', 'لفه': 'lafa',
    'mold': 'mold', 'قالب': 'mold',
    'tiplate': 'tiplate', 'صفيح': 'tiplate', 'plate': 'tiplate',
    'bottle': 'bottle', 'ازازه': 'bottle',
    'bucket': 'bucket', 'جردل': 'bucket',
    'scratch card': 'scratch_card', 'كارت': 'scratch_card',
    'dozen': 'dozen', 'دسته': 'dozen',
    'kg': 'kg',
    'نص شرينك': 'half_shrink', 'ربع شرينك': 'quarter_shrink',
    '10 كارت': 'scratch_card_10',
    'شنطه': 'bag_large',
}


def normalize_unit_type(unit_text):
    """Map any unit type text to a canonical form."""
    if not isinstance(unit_text, str):
        return 'unknown'
    normalized = normalize_arabic(unit_text.strip().lower())
    return UNIT_TYPE_CANONICAL.get(normalized, normalized)


# Apply to both datasets
maxab['unit_canonical'] = maxab['PACKING_UNIT_NAME_AR'].apply(
    lambda x: normalize_unit_type(x)
).combine_first(
    maxab['PACKING_UNIT_NAME_EN'].apply(lambda x: normalize_unit_type(x))
)

scrapped['unit_canonical'] = scrapped['UNIT_TYPE'].apply(normalize_unit_type)

print("Maxab unit mapping:")
print(maxab.groupby(['PACKING_UNIT_NAME_AR', 'PACKING_UNIT_NAME_EN', 'unit_canonical']).size().reset_index(name='count').to_string())
print("\nScrapped unit mapping:")
print(scrapped.groupby(['UNIT_TYPE', 'unit_canonical']).size().reset_index(name='count').to_string())

## Preprocess Both Datasets

In [ ]:
# Normalize text columns
maxab['sku_norm'] = maxab['SKU'].apply(normalize_arabic)
maxab['brand_norm'] = maxab['BRAND'].apply(normalize_arabic)
maxab['per_unit_price'] = maxab['CURRENT_PRICE'] / maxab['BASIC_UNIT_COUNT']

scrapped['product_norm'] = scrapped['SCRAPPED_PRODUCT'].apply(normalize_arabic)
scrapped['brand_norm'] = scrapped['SCRAPPED_BRAND'].apply(normalize_arabic)
scrapped['per_unit_price'] = np.where(
    scrapped['QUANTITY_PER_UNIT'].notna() & (scrapped['QUANTITY_PER_UNIT'] > 0),
    scrapped['SCRAPPED_PRICE'] / scrapped['QUANTITY_PER_UNIT'],
    np.nan
)

# Build maxab brand lookup: brand_norm -> list of product_ids
maxab_brands = sorted(maxab['brand_norm'].unique().tolist())
maxab_brand_set = set(maxab_brands)

# Build maxab product-level dedup (unique products with their SKU names)
maxab_products = maxab.drop_duplicates(subset='PRODUCT_ID')[['PRODUCT_ID', 'SKU', 'BRAND', 'CAT', 'sku_norm', 'brand_norm']].copy()
maxab_products = maxab_products.set_index('PRODUCT_ID')

print(f"Maxab unique products: {len(maxab_products)}")
print(f"Maxab unique brands (normalized): {len(maxab_brands)}")
print(f"\nSample normalized SKUs:")
for _, row in maxab_products.head(5).iterrows():
    print(f"  {row['SKU']}")
    print(f"  → {row['sku_norm']}")
print(f"\nSample normalized scrapped products:")
for _, row in scrapped.head(5).iterrows():
    print(f"  [{row['SOURCE_APP']}] {row['SCRAPPED_PRODUCT']}")
    print(f"  → {row['product_norm']}")

## Brand Matching

In [ ]:
# Pre-build brand token index for fast lookup
# Maps each single normalized token → set of full brand names containing it
brand_token_index = defaultdict(set)
for brand in maxab_brands:
    for token in brand.split():
        if len(token) >= 2:
            brand_token_index[token].add(brand)

# Pre-sort brands by length descending so longer (more specific) brands match first
maxab_brands_by_length = sorted(maxab_brands, key=len, reverse=True)

# Brand to product IDs lookup
brand_to_products = maxab.groupby('brand_norm')['PRODUCT_ID'].apply(set).to_dict()


def match_brand_tawfeer(scrapped_brand_norm):
    """For Tawfeer: strict brand matching with token-level containment.
    1. Exact match
    2. Token-level subset: فاتيكا tokens ⊂ فاتيكا كريم tokens (whole-word match)
    3. 90%+ fuzzy match for spelling variations
    Avoids character-level substring which causes false positives like تيكا ⊂ فاتيكا.
    """
    if not scrapped_brand_norm:
        return []
    
    if scrapped_brand_norm in maxab_brand_set:
        return [(scrapped_brand_norm, 100)]
    
    results = []
    scrapped_tokens = set(scrapped_brand_norm.split())
    
    for brand in maxab_brands:
        brand_tokens = set(brand.split())
        if scrapped_tokens.issubset(brand_tokens) or brand_tokens.issubset(scrapped_tokens):
            results.append((brand, 95))
    
    if not results:
        best = process.extract(scrapped_brand_norm, maxab_brands, scorer=fuzz.ratio, limit=3)
        for brand, score, _ in best:
            if score >= 90:
                results.append((brand, score))
    
    return results


# Category/product-type words that can precede a brand at position 0
# Built from maxab CAT column + common first tokens in SKU names that aren't brands
CATEGORY_WORDS = {
    'مكرونه', 'بسكويت', 'جبنه', 'حفاضات', 'مناديل', 'نعناع', 'بن', 'مربي',
    'كريم', 'لبان', 'ارز', 'ملح', 'تونه', 'صابون', 'جيل', 'مسحوق', 'سائل',
    'شامبو', 'شاي', 'عصير', 'زيت', 'لبن', 'زبادي', 'زبده', 'سمن', 'سمنه',
    'دقيق', 'سكر', 'شوكولاته', 'شيكولاته', 'كاكاو', 'قهوه', 'كابتشينو',
    'كوفي', 'شيبسي', 'ويفر', 'كيك', 'كورن', 'نودلز', 'صلصه', 'صوص',
    'كاتشب', 'مايونيز', 'خل', 'توابل', 'فلفل', 'كمون', 'عسل', 'حلاوه',
    'طحينه', 'طحينيه', 'فول', 'حمص', 'عدس', 'لوبيا', 'فاصوليا', 'بسله',
    'باميه', 'ملوخيه', 'سردين', 'ماكريل', 'برجر', 'لانشون', 'سجق', 'هوت',
    'ناجتس', 'بطاطس', 'كفته', 'دجاج', 'لحم', 'سمك', 'كروت', 'شحن',
    'بطاريات', 'بطاريه', 'كبريت', 'فوط', 'مبلله', 'مشروب', 'مياه',
    'حشريه', 'مبيدات', 'منظف', 'منعم', 'مزيل', 'مبيضات', 'معطرات',
    'معجون', 'شاور', 'صبغه', 'حنه', 'ليفه', 'رول', 'كيس', 'علبه',
    'كارت', 'حلوي', 'حلويات', 'بقسماط', 'فريك', 'برغل', 'نشا', 'خميره',
    'فانيليا', 'مرقه', 'سبريد', 'كاسترد', 'كسترد', 'جيلي', 'تمر',
    'شطه', 'خضار', 'مستلزمات', 'ادوات',
}


def find_brand_in_text(product_text_norm):
    """For non-Tawfeer: find which maxab brand is at the start of the product name.
    - Position 0: always checked for brand
    - Position 1: only checked if position 0 is a known category word
      (prevents 'سي ويلز' matching brand 'ويلز' since 'سي' is not a category word)"""
    if not product_text_norm:
        return []
    
    all_tokens = product_text_norm.split()
    if not all_tokens:
        return []
    
    # Determine valid start positions for brand
    pos0_is_category = all_tokens[0] in CATEGORY_WORDS
    max_brand_start = 1 if pos0_is_category else 0
    
    results = []
    
    for brand in maxab_brands_by_length:
        brand_tokens = brand.split()
        n = len(brand_tokens)
        
        if n == 1 and len(brand_tokens[0]) <= 2:
            continue
        
        for start in range(max_brand_start + 1):
            end = start + n
            if end > len(all_tokens):
                continue
            segment = all_tokens[start:end]
            if all(bt == st for bt, st in zip(brand_tokens, segment)):
                results.append((brand, 100 if start == 0 else 95))
                break
            if n >= 2 and start == 0:
                if fuzz.ratio(brand_tokens[0], all_tokens[0]) >= 85:
                    if all(any(fuzz.ratio(bt, all_tokens[i]) >= 85 for i in range(min(n+1, len(all_tokens)))) for bt in brand_tokens):
                        results.append((brand, 90))
                        break
    
    # Fallback: single-token brand at position 0 or 1 (if pos 0 is category)
    if not results:
        for pos in range(max_brand_start + 1):
            if pos < len(all_tokens) and len(all_tokens[pos]) >= 3 and all_tokens[pos] in maxab_brand_set:
                results.append((all_tokens[pos], 90 if pos == 0 else 85))
                break
    
    # Deduplicate: remove brands whose tokens are a proper subset of another found brand
    if len(results) > 1:
        brand_token_sets = [(b, set(b.split()), s) for b, s in results]
        filtered = []
        for b, bts, score in brand_token_sets:
            is_subset = any(bts < other_bts for _, other_bts, _ in brand_token_sets if other_bts != bts)
            if not is_subset:
                filtered.append((b, score))
        results = filtered if filtered else results
    
    return results[:5]


# Test brand matching
print("=== Tawfeer brand matching test ===")
tawfeer_test_brands = ['برسيل', 'فاين', 'تايد', 'اوكسي', 'كلوريل']
for b in tawfeer_test_brands:
    matches = match_brand_tawfeer(normalize_arabic(b))
    print(f"  {b} → {matches}")

print("\n=== Brand-in-text matching test ===")
text_tests = [
    "اندومي بطعم الفراخ + 4 اكياس مجانا - 70 جم",
    "سيترس منظف برائحة الليمون - 1 لتر",
    "تايد اوتوماتيك برائحة داوني - 900 جم",
    "ماريو تونة مفتتة باردة - 140 جم",
    "اوريو بسكويت شيكولاتة محشو كريمة شيكولاتة - 3 قطع",
]
for t in text_tests:
    matches = find_brand_in_text(normalize_arabic(t))
    print(f"  {t}")
    print(f"    → {matches}")

## SKU Matching & Price Validation

In [ ]:
def compute_text_similarity(text_a, text_b):
    """Multi-signal text similarity optimized for Arabic product names."""
    if not text_a or not text_b:
        return 0.0
    
    token_sort = fuzz.token_sort_ratio(text_a, text_b)
    token_set = fuzz.token_set_ratio(text_a, text_b)
    partial = fuzz.partial_ratio(text_a, text_b)
    
    return max(token_sort, token_set * 0.95, partial * 0.85)


def price_compatible(maxab_row, scrapped_row, threshold=0.20):
    """Check price compatibility across different unit levels.
    
    Tries multiple normalization strategies:
    1. Direct pack-level comparison (if unit types match)
    2. Per-unit price comparison (both normalized to single unit)
    3. Cross-level inference (scrapped pack vs maxab per-unit or vice versa)
    
    Returns (is_compatible, best_price_diff_pct, match_description)
    """
    m_price = maxab_row['CURRENT_PRICE']
    m_units = maxab_row['BASIC_UNIT_COUNT']
    m_per_unit = m_price / m_units if m_units > 0 else m_price
    m_unit_type = maxab_row['unit_canonical']
    
    s_price = scrapped_row['SCRAPPED_PRICE']
    s_qty = scrapped_row.get('QUANTITY_PER_UNIT', np.nan)
    s_per_unit = s_price / s_qty if pd.notna(s_qty) and s_qty > 0 else np.nan
    s_unit_type = scrapped_row['unit_canonical']
    
    comparisons = []
    
    # Strategy 1: Direct comparison if unit types are compatible
    if m_unit_type == s_unit_type:
        if m_price > 0:
            diff = abs(m_price - s_price) / m_price
            comparisons.append((diff, 'direct_same_unit'))
    
    # Strategy 2: Per-unit comparison (both sides normalized)
    if pd.notna(s_per_unit) and m_per_unit > 0:
        diff = abs(m_per_unit - s_per_unit) / m_per_unit
        comparisons.append((diff, 'per_unit'))
    
    # Strategy 3: Scrapped total vs maxab total (different unit counts, same unit type family)
    if m_unit_type == s_unit_type and pd.notna(s_qty) and m_units > 0:
        if m_units == s_qty and m_price > 0:
            diff = abs(m_price - s_price) / m_price
            comparisons.append((diff, 'same_qty_same_unit'))
    
    # Strategy 4: Scrapped price might be per-piece, maxab is per-pack
    if m_per_unit > 0 and s_unit_type in ('piece', 'item', 'قطعه'):
        diff = abs(m_per_unit - s_price) / m_per_unit
        comparisons.append((diff, 'scrapped_piece_vs_maxab_perunit'))
    
    # Strategy 5: Maxab is per-piece, scrapped is per-pack
    if m_units == 1 and pd.notna(s_qty) and s_qty > 1 and s_per_unit > 0:
        diff = abs(m_price - s_per_unit) / m_price if m_price > 0 else 1.0
        comparisons.append((diff, 'maxab_piece_vs_scrapped_perunit'))
    
    # Strategy 6: Try matching maxab price to scrapped total with ratio inference
    if m_price > 0 and s_price > 0:
        ratio = s_price / m_price
        for mult in [1, 0.5, 2, 0.25, 4, 3, 1/3]:
            if abs(ratio - mult) / max(mult, 0.01) < threshold:
                comparisons.append((abs(ratio - mult) / max(mult, 0.01), f'ratio_{mult:.2f}'))
                break
    
    if not comparisons:
        return False, 1.0, 'no_comparison'
    
    best = min(comparisons, key=lambda x: x[0])
    return best[0] <= threshold, best[0], best[1]


# Quick price test
print("Price compatibility test:")
test_maxab = {'CURRENT_PRICE': 385.0, 'BASIC_UNIT_COUNT': 1, 'unit_canonical': 'carton'}
test_scrapped = {'SCRAPPED_PRICE': 287.0, 'QUANTITY_PER_UNIT': np.nan, 'unit_canonical': 'carton'}
ok, diff, desc = price_compatible(test_maxab, test_scrapped)
print(f"  Maxab carton@385 vs Scrapped carton@287: compatible={ok}, diff={diff:.2%}, strategy={desc}")

test_maxab2 = {'CURRENT_PRICE': 1872.0, 'BASIC_UNIT_COUNT': 24, 'unit_canonical': 'carton'}
test_scrapped2 = {'SCRAPPED_PRICE': 78.0, 'QUANTITY_PER_UNIT': np.nan, 'unit_canonical': 'piece'}
ok2, diff2, desc2 = price_compatible(test_maxab2, test_scrapped2)
print(f"  Maxab carton(24)@1872 vs Scrapped piece@78: compatible={ok2}, diff={diff2:.2%}, strategy={desc2}")

## Main Matching Pipeline

In [ ]:
def get_candidate_products(scrapped_row, top_n=10):
    """Get candidate maxab products for a scrapped row using brand + text similarity."""
    app = scrapped_row['SOURCE_APP']
    product_norm = scrapped_row['product_norm']
    brand_norm = scrapped_row['brand_norm']
    
    candidate_product_ids = set()
    brand_score_map = {}
    
    if app == 'Tawfeer' and brand_norm:
        brand_matches = match_brand_tawfeer(brand_norm)
        for matched_brand, score in brand_matches:
            if matched_brand in brand_to_products:
                pids = brand_to_products[matched_brand]
                candidate_product_ids.update(pids)
                for pid in pids:
                    brand_score_map[pid] = max(brand_score_map.get(pid, 0), score)
    else:
        brand_matches = find_brand_in_text(product_norm)
        for matched_brand, score in brand_matches:
            if matched_brand in brand_to_products:
                pids = brand_to_products[matched_brand]
                candidate_product_ids.update(pids)
                for pid in pids:
                    brand_score_map[pid] = max(brand_score_map.get(pid, 0), score)
    
    if not candidate_product_ids:
        return []
    
    candidates = maxab_products.loc[maxab_products.index.isin(candidate_product_ids)]
    
    # For Tawfeer: compare brand+product against maxab SKU
    if app == 'Tawfeer' and brand_norm:
        search_text = f"{brand_norm} {product_norm}"
    else:
        search_text = product_norm
    
    scored = []
    for pid, row in candidates.iterrows():
        text_sim = compute_text_similarity(search_text, row['sku_norm'])
        
        size_ok = sizes_compatible(search_text, row['sku_norm'], tolerance=0.15)
        if size_ok is False:
            continue
        
        # Lower brand weight when brand is inferred from text (no explicit brand column)
        has_explicit_brand = app == 'Tawfeer' and brand_norm
        brand_weight = 0.15 if has_explicit_brand else 0.05
        brand_bonus = brand_score_map.get(pid, 0) * brand_weight
        size_bonus = 10 if size_ok is True else 0
        
        combined = text_sim + brand_bonus + size_bonus
        scored.append((pid, combined, text_sim, brand_score_map.get(pid, 0)))
    
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]


def match_packing_unit(maxab_product_rows, scrapped_row):
    """Given matched maxab product (all its packing units), find the best unit match."""
    results = []
    for _, m_row in maxab_product_rows.iterrows():
        unit_match = m_row['unit_canonical'] == scrapped_row['unit_canonical']
        price_ok, price_diff, price_strategy = price_compatible(m_row, scrapped_row)
        
        unit_score = 30 if unit_match else 0
        price_score = max(0, 30 * (1 - price_diff)) if price_ok else 0
        
        results.append({
            'PRODUCT_ID': m_row['PRODUCT_ID'],
            'SKU': m_row['SKU'],
            'BRAND': m_row['BRAND'],
            'CAT': m_row['CAT'],
            'CURRENT_PRICE': m_row['CURRENT_PRICE'],
            'BASIC_UNIT_COUNT': m_row['BASIC_UNIT_COUNT'],
            'PACKING_UNIT_NAME_EN': m_row['PACKING_UNIT_NAME_EN'],
            'PACKING_UNIT_NAME_AR': m_row['PACKING_UNIT_NAME_AR'],
            'unit_match': unit_match,
            'price_compatible': price_ok,
            'price_diff_pct': price_diff,
            'price_strategy': price_strategy,
            'unit_price_score': unit_score + price_score,
        })
    
    results.sort(key=lambda x: x['unit_price_score'], reverse=True)
    return results


def map_single_scrapped(scrapped_row, min_text_score=45):
    """Full mapping pipeline for a single scrapped row."""
    candidates = get_candidate_products(scrapped_row)
    
    if not candidates:
        return []
    
    all_matches = []
    for pid, combined_score, text_sim, brand_sim in candidates:
        if text_sim < min_text_score:
            continue
        
        product_rows = maxab[maxab['PRODUCT_ID'] == pid]
        unit_matches = match_packing_unit(product_rows, scrapped_row)
        
        for um in unit_matches:
            um['text_similarity'] = text_sim
            um['brand_similarity'] = brand_sim
            um['combined_score'] = combined_score + um['unit_price_score']
            um['SOURCE_APP'] = scrapped_row['SOURCE_APP']
            um['SUPPLIER'] = scrapped_row['SUPPLIER']
            um['SCRAPPED_PRODUCT'] = scrapped_row['SCRAPPED_PRODUCT']
            um['SCRAPPED_BRAND'] = scrapped_row.get('SCRAPPED_BRAND', '')
            um['SCRAPPED_PRICE'] = scrapped_row['SCRAPPED_PRICE']
            um['QUANTITY_PER_UNIT'] = scrapped_row.get('QUANTITY_PER_UNIT', np.nan)
            um['SCRAPPED_UNIT_TYPE'] = scrapped_row['UNIT_TYPE']
            all_matches.append(um)
    
    all_matches.sort(key=lambda x: x['combined_score'], reverse=True)
    return all_matches

print("Pipeline functions ready.")

## Run Matching

In [ ]:
from datetime import datetime

print(f"Starting mapping at {datetime.now().strftime('%H:%M:%S')}")
print(f"Processing {len(scrapped)} scrapped rows...")

all_results = []
unmatched = []
match_counts = {'total': 0, 'matched': 0, 'unmatched': 0}

for idx, s_row in scrapped.iterrows():
    matches = map_single_scrapped(s_row, min_text_score=45)
    match_counts['total'] += 1
    
    if matches:
        best = matches[0]
        all_results.append(best)
        match_counts['matched'] += 1
        
        # Also keep secondary matches (one-to-many for soda/scent variants)
        for m in matches[1:]:
            if m['combined_score'] >= best['combined_score'] * 0.90:
                all_results.append(m)
    else:
        unmatched.append({
            'SOURCE_APP': s_row['SOURCE_APP'],
            'SCRAPPED_PRODUCT': s_row['SCRAPPED_PRODUCT'],
            'SCRAPPED_BRAND': s_row.get('SCRAPPED_BRAND', ''),
            'SCRAPPED_PRICE': s_row['SCRAPPED_PRICE'],
            'SCRAPPED_UNIT_TYPE': s_row['UNIT_TYPE'],
        })
        match_counts['unmatched'] += 1
    
    if (idx + 1) % 500 == 0:
        print(f"  Processed {idx+1}/{len(scrapped)} ({match_counts['matched']}/{match_counts['total']} matched)")

results_df = pd.DataFrame(all_results)
unmatched_df = pd.DataFrame(unmatched)

print(f"\nDone at {datetime.now().strftime('%H:%M:%S')}")
print(f"Total: {match_counts['total']}, Matched: {match_counts['matched']} ({100*match_counts['matched']/match_counts['total']:.1f}%), Unmatched: {match_counts['unmatched']}")
print(f"Result rows (incl. secondary matches): {len(results_df)}")

## Results Analysis

In [ ]:
if len(results_df) > 0:
    display_cols = [
        'SOURCE_APP', 'SCRAPPED_PRODUCT', 'SCRAPPED_BRAND', 'SCRAPPED_PRICE', 'SCRAPPED_UNIT_TYPE',
        'PRODUCT_ID', 'SKU', 'BRAND', 'CURRENT_PRICE', 'PACKING_UNIT_NAME_EN', 'BASIC_UNIT_COUNT',
        'text_similarity', 'brand_similarity', 'price_compatible', 'price_diff_pct', 'combined_score'
    ]
    display_cols = [c for c in display_cols if c in results_df.columns]
    
    print("=== Match Rate by App ===")
    app_stats = results_df.groupby('SOURCE_APP').agg(
        matches=('PRODUCT_ID', 'count'),
        avg_text_sim=('text_similarity', 'mean'),
        avg_combined=('combined_score', 'mean'),
        price_compat_rate=('price_compatible', 'mean'),
    ).round(2)
    print(app_stats.to_string())
    
    print("\n=== Score Distribution ===")
    print(results_df['text_similarity'].describe().round(1).to_string())
    
    print("\n=== Sample HIGH confidence matches (text_sim >= 75) ===")
    high = results_df[results_df['text_similarity'] >= 75].sort_values('combined_score', ascending=False)
    if len(high) > 0:
        for _, r in high.head(15).iterrows():
            print(f"\n  [{r['SOURCE_APP']}] {r['SCRAPPED_PRODUCT']}")
            if pd.notna(r.get('SCRAPPED_BRAND')) and r.get('SCRAPPED_BRAND'):
                print(f"    Scrapped brand: {r['SCRAPPED_BRAND']}")
            print(f"    → Maxab: {r['SKU']} ({r['BRAND']})")
            print(f"    Price: {r['SCRAPPED_PRICE']:.1f} vs {r['CURRENT_PRICE']:.1f} | Unit: {r['SCRAPPED_UNIT_TYPE']} vs {r['PACKING_UNIT_NAME_EN']}({r['BASIC_UNIT_COUNT']})")
            print(f"    Scores: text={r['text_similarity']:.0f}, brand={r['brand_similarity']:.0f}, price_ok={r['price_compatible']}, diff={r['price_diff_pct']:.1%}")
    else:
        print("  No high confidence matches found.")
    
    print("\n=== Sample LOW confidence matches (text_sim < 55) ===")
    low = results_df[results_df['text_similarity'] < 55].sort_values('combined_score', ascending=False)
    if len(low) > 0:
        for _, r in low.head(10).iterrows():
            print(f"\n  [{r['SOURCE_APP']}] {r['SCRAPPED_PRODUCT']}")
            if pd.notna(r.get('SCRAPPED_BRAND')) and r.get('SCRAPPED_BRAND'):
                print(f"    Scrapped brand: {r['SCRAPPED_BRAND']}")
            print(f"    → Maxab: {r['SKU']} ({r['BRAND']})")
            print(f"    Scores: text={r['text_similarity']:.0f}, brand={r['brand_similarity']:.0f}, combined={r['combined_score']:.0f}")
    else:
        print("  None.")
else:
    print("No matches found.")

In [ ]:
# Unmatched products breakdown
if len(unmatched_df) > 0:
    print("=== Unmatched by App ===")
    print(unmatched_df['SOURCE_APP'].value_counts().to_string())
    
    print("\n=== Sample Unmatched Products ===")
    for app in unmatched_df['SOURCE_APP'].unique():
        print(f"\n--- {app} ---")
        for _, r in unmatched_df[unmatched_df['SOURCE_APP']==app].head(5).iterrows():
            brand_info = f" (brand: {r['SCRAPPED_BRAND']})" if pd.notna(r.get('SCRAPPED_BRAND')) and r.get('SCRAPPED_BRAND') else ""
            print(f"  {r['SCRAPPED_PRODUCT']}{brand_info} @ {r['SCRAPPED_PRICE']}")
else:
    print("All products matched!")

## Export

In [ ]:
export_cols = [
    'SOURCE_APP', 'SUPPLIER', 'SCRAPPED_PRODUCT', 'SCRAPPED_BRAND', 'SCRAPPED_PRICE', 
    'QUANTITY_PER_UNIT', 'SCRAPPED_UNIT_TYPE',
    'PRODUCT_ID', 'SKU', 'BRAND', 'CAT', 'CURRENT_PRICE', 'BASIC_UNIT_COUNT',
    'PACKING_UNIT_NAME_AR', 'PACKING_UNIT_NAME_EN',
    'text_similarity', 'brand_similarity', 'price_compatible', 'price_diff_pct', 
    'price_strategy', 'combined_score'
]
export_cols = [c for c in export_cols if c in results_df.columns]

results_export = results_df[export_cols].sort_values(['SOURCE_APP', 'combined_score'], ascending=[True, False])
results_export.to_excel(r'd:\scripts\Mustafa\Mapping\mapping_results.xlsx', index=False)

if len(unmatched_df) > 0:
    unmatched_df.to_excel(r'd:\scripts\Mustafa\Mapping\mapping_unmatched.xlsx', index=False)

print(f"Exported {len(results_export)} matched rows to mapping_results.xlsx")
print(f"Exported {len(unmatched_df)} unmatched rows to mapping_unmatched.xlsx")